In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report
from datasets import load_dataset, concatenate_datasets

tf.keras.utils.set_random_seed(42)
print("Python:", sys.version)
print("TensorFlow:", tf.__version__)


In [ ]:
# (opcional) quita warning de symlinks en Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

ds = load_dataset(
    "neonwatty/amazon_reviews_multi",
    data_dir="es",
    revision="refs/convert/parquet"
)
ds


In [ ]:
def add_text_label(ex):
    title = ex.get("review_title") or ""
    body  = ex.get("review_body") or ""
    text = (str(title) + " " + str(body)).strip()

    stars = int(ex["stars"])
    if stars <= 2:
        label = 0  # negativo
    elif stars >= 4:
        label = 1  # positivo
    else:
        label = -1 # neutral (3)

    return {"text": text, "label": label}

ds_proc = {}
for split in ["train", "validation", "test"]:
    tmp = ds[split].map(add_text_label)
    tmp = tmp.filter(lambda x: x["label"] != -1 and len(x["text"]) > 20)  # filtra neutrales y textos muy cortos
    ds_proc[split] = tmp

ds_proc["train"], ds_proc["validation"], ds_proc["test"]


In [ ]:
def balance_split(hfds, n_each, seed=42):
    neg = hfds.filter(lambda x: x["label"] == 0).shuffle(seed=seed).select(range(n_each))
    pos = hfds.filter(lambda x: x["label"] == 1).shuffle(seed=seed).select(range(n_each))
    return concatenate_datasets([neg, pos]).shuffle(seed=seed)

# Tamaños (ajusta si tu PC es lenta)
train_hf = balance_split(ds_proc["train"], n_each=20000)       # 40k total
val_hf   = balance_split(ds_proc["validation"], n_each=2500)   # 5k total
test_hf  = balance_split(ds_proc["test"], n_each=2500)         # 5k total

print("Train balance:", np.bincount(train_hf["label"]))
print("Val balance:", np.bincount(val_hf["label"]))
print("Test balance:", np.bincount(test_hf["label"]))


In [ ]:
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def hf_to_tfds(hfds, shuffle=True):
    x = np.array(hfds["text"], dtype=str)
    y = np.array(hfds["label"], dtype=np.int32)
    ds_tf = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds_tf = ds_tf.shuffle(buffer_size=min(len(x), 20000), seed=42)
    return ds_tf.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = hf_to_tfds(train_hf, shuffle=True)
val_ds   = hf_to_tfds(val_hf, shuffle=False)
test_ds  = hf_to_tfds(test_hf, shuffle=False)

# ver un ejemplo
for x_batch, y_batch in train_ds.take(1):
    print("x shape:", x_batch.shape, "dtype:", x_batch.dtype)
    print("y shape:", y_batch.shape, "unique labels:", np.unique(y_batch.numpy()))
    print("ejemplo texto:", x_batch[0].numpy().decode("utf-8")[:200], "...")


In [ ]:
VOCAB_SIZE = 20000
SEQ_LEN = 250

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQ_LEN
)

text_only_train = train_ds.unbatch().map(lambda x, y: x)
vectorizer.adapt(text_only_train.batch(256))

print("Vectorizer listo ✅")


In [ ]:
EMBED_DIM = 128
RNN_UNITS = 64

def build_model(kind="gru"):
    inputs = tf.keras.Input(shape=(), dtype=tf.string)
    x = vectorizer(inputs)
    x = tf.keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True)(x)

    if kind.lower() == "lstm":
        x = tf.keras.layers.LSTM(RNN_UNITS)(x)
    elif kind.lower() == "gru":
        x = tf.keras.layers.GRU(RNN_UNITS)(x)
    else:
        raise ValueError("kind debe ser 'lstm' o 'gru'")

    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    opt = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    return model


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
]

lstm_model = build_model("lstm")
history_lstm = lstm_model.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=callbacks)

lstm_loss, lstm_acc = lstm_model.evaluate(test_ds, verbose=0)
print("LSTM Test Acc:", lstm_acc)
print("LSTM Test Loss:", lstm_loss)


In [ ]:
gru_model = build_model("gru")
history_gru = gru_model.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=callbacks)

gru_loss, gru_acc = gru_model.evaluate(test_ds, verbose=0)
print("GRU Test Acc:", gru_acc)
print("GRU Test Loss:", gru_loss)


In [ ]:
results = pd.DataFrame([
    {"Modelo":"LSTM", "Test_Accuracy":lstm_acc, "Test_Loss":lstm_loss},
    {"Modelo":"GRU",  "Test_Accuracy":gru_acc,  "Test_Loss":gru_loss},
]).sort_values("Test_Accuracy", ascending=False)

results


In [ ]:
def plot_history(hist, title=""):
    plt.figure()
    plt.plot(hist.history["accuracy"], label="train_acc")
    plt.plot(hist.history["val_accuracy"], label="val_acc")
    plt.title(f"Accuracy - {title}")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy")
    plt.legend(); plt.show()

    plt.figure()
    plt.plot(hist.history["loss"], label="train_loss")
    plt.plot(hist.history["val_loss"], label="val_loss")
    plt.title(f"Loss - {title}")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend(); plt.show()

plot_history(history_lstm, "LSTM")
plot_history(history_gru, "GRU")


In [ ]:
def get_preds(model, ds_tf):
    y_true, y_prob = [], []
    for x_batch, y_batch in ds_tf:
        probs = model.predict(x_batch, verbose=0).ravel()
        y_prob.extend(probs.tolist())
        y_true.extend(y_batch.numpy().tolist())
    y_true = np.array(y_true)
    y_pred = (np.array(y_prob) >= 0.5).astype(int)
    return y_true, y_pred

best_model = gru_model if gru_acc >= lstm_acc else lstm_model

y_true, y_pred = get_preds(best_model, test_ds)
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nReport:\n", classification_report(y_true, y_pred))


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

display(HTML("""
<style>
.card{border:1px solid #00000018;border-radius:14px;padding:16px;max-width:900px;background:#fff;
box-shadow:0 6px 18px rgba(0,0,0,.08);font-family:Arial}
.title{font-size:18px;font-weight:700;margin:0 0 6px 0}
.subtitle{color:#555;margin:0 0 12px 0}
.badge{display:inline-block;padding:6px 10px;border-radius:999px;font-weight:700;font-size:12px;margin-right:10px}
.pos{background:#e8fff0;color:#0b6b2a;border:1px solid #0b6b2a20}
.neg{background:#ffecec;color:#8a0e0e;border:1px solid #8a0e0e20}
.neu{background:#f2f2f2;color:#444;border:1px solid #4442}
</style>
"""))

review_box = widgets.Textarea(
    value="",
    placeholder="Escribe una reseña en español...\nEj: 'Excelente calidad, llegó rápido y funciona perfecto.'",
    description="Reseña:",
    style={"description_width":"initial"},
    layout=widgets.Layout(width="900px", height="140px")
)

btn_predict = widgets.Button(description="Analizar", button_style="success", icon="search",
                             layout=widgets.Layout(width="160px", height="40px"))
btn_clear = widgets.Button(description="Limpiar", icon="trash",
                           layout=widgets.Layout(width="140px", height="40px"))

out = widgets.Output()

def on_predict(_):
    text = review_box.value.strip()
    with out:
        clear_output()
        if len(text) < 10:
            display(HTML('<div class="card"><div class="title">⚠️ Escribe una reseña más larga</div></div>'))
            return
        prob = float(best_model.predict(tf.constant([text]), verbose=0)[0][0])
        label = "POSITIVA" if prob >= 0.5 else "NEGATIVA"
        cls = "pos" if label == "POSITIVA" else "neg"
        display(HTML(f"""
        <div class="card">
          <div class="title">Resultado</div>
          <div class="subtitle">Clasificación binaria (español): positiva vs negativa</div>
          <div>
            <span class="badge {cls}">{label}</span>
            <span class="badge neu">Prob. positiva: {prob:.3f}</span>
          </div>
        </div>
        """))

def on_clear(_):
    review_box.value = ""
    with out: clear_output()

btn_predict.on_click(on_predict)
btn_clear.on_click(on_clear)

display(widgets.VBox([
    widgets.HTML('<div class="card"><div class="title">Formulario: Sentimiento en Español</div><div class="subtitle">Escribe una reseña y presiona Analizar.</div></div>'),
    widgets.HBox([btn_predict, btn_clear]),
    review_box,
    out
]))
